# mrv-lib Quickstart: Representation Invariance Test

This notebook demonstrates how to use mrv-lib to check whether your regime model
produces **stable labels** across different feature representations.

**The problem:** You fit a GMM/HMM regime model using volatility, drawdown, VaR, etc.
But what happens if you swap one factor for another? If the regime labels change
significantly, your downstream risk decisions are sitting on an arbitrary modelling choice.

**What this test does:** Fit the same model on multiple factor sets, then measure
pairwise Adjusted Rand Index (ARI) to quantify label agreement.

---

## Setup

```bash
pip install mrv-lib[all]
```

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mrv.data.factors import build_factors, log_returns, volatility
from mrv.data.normalize import normalize
from mrv.models import fit
from mrv.validator.metrics import ari, ordering_consistency, ARI_THRESHOLD, SPEARMAN_THRESHOLD

## 1. Load price data

We use synthetic data here so the notebook runs without an IB connection.
In practice, replace this with your own price series or use `mrv.pipeline.download()`.

In [ ]:
np.random.seed(42)
n = 1000

# Simulate a price series with regime-like behavior
# Low-vol regime (60%), high-vol regime (40%)
regime = np.random.choice([0, 1], size=n, p=[0.6, 0.4])
returns = np.where(regime == 0,
                   np.random.randn(n) * 0.005 + 0.0003,   # calm
                   np.random.randn(n) * 0.02 - 0.0005)    # stressed
price = 100.0 * np.exp(np.cumsum(returns))
price = pd.Series(price, index=pd.bdate_range("2018-01-01", periods=n), name="SyntheticAsset")

print(f"Price series: {len(price)} observations")
print(f"Range: {price.index[0].date()} to {price.index[-1].date()}")
price.plot(title="Synthetic price series", figsize=(10, 3))
plt.show()

## 2. Compute risk factors

mrv-lib provides 7 built-in factors. We define **three different factor sets**
to test whether the regime labels are representation-invariant.

| Set | Factors | Rationale |
|-----|---------|----------|
| A | vol, drawdown, maxdd, VaR, CVaR | Full tail-risk set |
| B | vol, drawdown, VaR, CVaR | Drop max drawdown |
| C | realized_skew, stability, VaR, CVaR | Replace level factors with higher-order moments |

In [ ]:
factor_sets = {
    "A: vol+dd+maxdd+var+cvar": ["volatility", "drawdown", "max_drawdown_window", "var", "cvar"],
    "B: vol+dd+var+cvar":       ["volatility", "drawdown", "var", "cvar"],
    "C: skew+stab+var+cvar":    ["realized_skew", "stability", "var", "cvar"],
}

cfg = {"factors": {"vol_window": 20, "drawdown_window": 60, "tail_window": 60,
                   "tail_alpha": 0.05, "skew_window": 60, "stability_window": 60},
       "normalize": {"mode": "rolling_zscore", "window": 120}}

factor_dfs = {}
for label, factors in factor_sets.items():
    raw = build_factors(price, factors=factors, cfg=cfg)
    normed = normalize(raw, cfg=cfg).dropna()
    factor_dfs[label] = normed
    print(f"{label}: {normed.shape[0]} obs, {normed.shape[1]} features")

## 3. Fit regime model on each factor set

We fit a 3-state GMM on each factor set independently. If the regime model is
representation-invariant, all three should produce similar labels.

In [ ]:
labels = {}
for label, df in factor_dfs.items():
    result = fit(df, model="gmm", n_states=3)
    labels[label] = result
    unique, counts = np.unique(result, return_counts=True)
    dist = ", ".join(f"state {s}: {c} ({c/len(result)*100:.0f}%)" for s, c in zip(unique, counts))
    print(f"{label}")
    print(f"  {dist}")

## 4. Compare labels: ARI matrix

The **Adjusted Rand Index (ARI)** measures agreement between two label sets:
- ARI = 1.0: perfect agreement
- ARI ~ 0.0: random agreement
- ARI < 0: worse than random

**Threshold: ARI >= 0.65** (Steinley 2004) indicates acceptable partition recovery.

In [ ]:
from itertools import combinations

set_names = list(labels.keys())
n = len(set_names)
ari_matrix = pd.DataFrame(np.eye(n), index=set_names, columns=set_names)

for (la, a), (lb, b) in combinations(labels.items(), 2):
    val = ari(a, b)
    ari_matrix.loc[la, lb] = ari_matrix.loc[lb, la] = val

print("Cross-Representation ARI Matrix:")
print(ari_matrix.round(3).to_string())
print(f"\nThreshold: {ARI_THRESHOLD}")
print(f"Mean off-diagonal ARI: {ari_matrix.values[np.triu_indices(n, k=1)].mean():.3f}")

## 5. Ordering consistency: Spearman correlation

Even if the exact partition boundaries differ, do the representations agree on
which observations are **riskier**? We measure this via Spearman rank correlation
of risk-ranked labels.

**Threshold: Spearman >= 0.85** indicates stable risk ordering.

In [ ]:
risk_proxy = volatility(log_returns(price), window=20, annualize=False).dropna().values

sp_matrix = pd.DataFrame(np.eye(n), index=set_names, columns=set_names)

for (la, a), (lb, b) in combinations(labels.items(), 2):
    nc = min(len(a), len(b), len(risk_proxy))
    val = ordering_consistency(a[:nc], b[:nc], risk_proxy[:nc])
    sp_matrix.loc[la, lb] = sp_matrix.loc[lb, la] = val

print("Ordering Consistency (Spearman) Matrix:")
print(sp_matrix.round(3).to_string())
print(f"\nThreshold: {SPEARMAN_THRESHOLD}")
print(f"Mean off-diagonal Spearman: {sp_matrix.values[np.triu_indices(n, k=1)].mean():.3f}")

## 6. Verdict

mrv-lib uses a **two-layer verdict**:

| Layer | Metric | Threshold | Question |
|-------|--------|-----------|----------|
| Partition stability | ARI | >= 0.65 | Are the labels the same? |
| Ordering stability | Spearman | >= 0.85 | Do they at least rank risk consistently? |

A model that **fails partition but passes ordering** means: the exact label
boundaries shift with representation, but the relative risk ranking is preserved.
This is less severe but still a material finding for SR 11-7 documentation.

In [ ]:
offdiag_ari = ari_matrix.values[np.triu_indices(n, k=1)]
offdiag_sp = sp_matrix.values[np.triu_indices(n, k=1)]

mean_ari = offdiag_ari.mean()
mean_sp = offdiag_sp.mean()

partition_pass = mean_ari >= ARI_THRESHOLD
ordering_pass = mean_sp >= SPEARMAN_THRESHOLD

print("=" * 50)
print("REPRESENTATION INVARIANCE VERDICT")
print("=" * 50)
print(f"Partition stability:  ARI = {mean_ari:.3f}  {'PASS' if partition_pass else 'FAIL'}  (threshold: {ARI_THRESHOLD})")
print(f"Ordering stability:   Spearman = {mean_sp:.3f}  {'PASS' if ordering_pass else 'FAIL'}  (threshold: {SPEARMAN_THRESHOLD})")
print("=" * 50)

if partition_pass and ordering_pass:
    print("\nConclusion: Regime labels are stable across representations.")
elif not partition_pass and ordering_pass:
    print("\nConclusion: Label boundaries shift, but risk ordering is preserved.")
    print("Recommendation: Document representation sensitivity in SR 11-7 filing.")
else:
    print("\nConclusion: Regime labels are NOT representation-invariant.")
    print("Recommendation: Investigate factor selection; consider ensemble or")
    print("representation-agnostic approaches before production deployment.")

## 7. Visualize: ARI heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
data = ari_matrix.values.astype(float)
im = ax.imshow(data, vmin=-0.1, vmax=1.0, cmap="RdYlGn", aspect="auto")
short_labels = ["Set A", "Set B", "Set C"]
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(short_labels)
ax.set_yticklabels(short_labels)
for i in range(n):
    for j in range(n):
        v = data[i, j]
        ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=12,
                fontweight="bold" if i != j else "normal",
                color="white" if v < 0.4 else "black")
ax.set_title(f"Cross-Representation ARI\n(threshold = {ARI_THRESHOLD})", fontweight="bold")
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("ARI")
cbar.ax.axhline(y=ARI_THRESHOLD, color="black", linewidth=1.5, linestyle="--")
fig.tight_layout()
plt.show()

## 8. Using the full pipeline (CLI & config)

For production use, mrv-lib provides a config-driven pipeline that handles
data loading, factor computation, model fitting, validation, and PDF report
generation in one command:

```bash
# CLI
python run.py run config.yaml rep
```

```python
# Python
from mrv.pipeline import run
run("config.yaml", "rep")
```

This produces a timestamped report directory with:
- `result.json` — full results (reusable)
- `report.pdf` — professional PDF with dashboard, heatmaps, and remediation plan
- `summary.txt` — plain text quick view
- `{asset}.png` — ARI heatmap per asset

See `config.yaml` for all available settings.